# Bitcoin Price Forecasting — Enhanced & Modernized (2018–2026)

**Improvements over the original notebook:**
1. **Data updated to June 2026** — includes the 2024–2025 bull run and current market.
2. **Richer feature engineering** — RSI, MACD, Bollinger Bands, ATR, OBV, volume ratio.
3. **New model: LightGBM** — gradient-boosted trees that are often faster and more accurate than XGBoost on tabular financial data.
4. **Stacking ensemble** — blends XGBoost + LightGBM predictions via a Ridge meta-learner.
5. **Expanded visualizations** — rolling 30-day MAE and side-by-side feature importance plots.
6. All original methodology fixes retained (single chronological test window, no data leakage, TimeSeriesSplit tuning).

| Model | Type |
|---|---|
| Naive (t-1) | Baseline |
| XGBoost | Tuned gradient-boosted trees |
| **LightGBM** *(new)* | Tuned gradient-boosted trees (leaf-wise, often faster & better) |
| **Stacking Ensemble** *(new)* | XGB + LGBM → Ridge meta-learner |
| Prophet | Additive time-series (multi-step, for reference) |
| LSTM | Deep learning (1-step walk-forward) |


## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit, cross_val_predict
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import Ridge

import xgboost as xgb
import lightgbm as lgb
from prophet import Prophet
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
print("All imports OK")

## 1. Data download (updated through June 2026)

We now pull full **OHLCV** data so we can compute volume-based features and volatility indicators.

In [ ]:
ticker     = "BTC-USD"
start_date = "2018-01-01"
end_date   = "2026-06-10"      # ← updated end date

df_raw = yf.download(ticker, start=start_date, end=end_date, auto_adjust=True)

# Flatten MultiIndex columns if present (yfinance >=0.2.x)
if isinstance(df_raw.columns, pd.MultiIndex):
    df_raw.columns = df_raw.columns.get_level_values(0)

df_raw = df_raw[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
df_raw.index = pd.to_datetime(df_raw.index)

print(f"Downloaded {len(df_raw):,} rows  |  {df_raw.index.min().date()} → {df_raw.index.max().date()}")
df_raw.tail(3)

## 2. Feature engineering

All indicators are shifted by ≥1 day before use to prevent look-ahead leakage.

**New technical indicators:**
- **RSI (14)** — relative strength; overbought/oversold signal
- **MACD** — trend momentum (EMA12 − EMA26)
- **Bollinger Band position** — price location within its 20-day ±2σ band
- **ATR (14)** — realized volatility proxy
- **OBV** — On-Balance Volume; accumulation/distribution
- **Volume ratio** — today's volume vs 20-day average


In [ ]:
feat = df_raw.copy()

# Price lags
for lag in [1, 3, 7, 14, 30]:
    feat[f'Lag{lag}'] = feat['Close'].shift(lag)

# Rolling statistics (shifted — never see today)
for w in [7, 14, 30]:
    feat[f'Roll_Mean_{w}'] = feat['Close'].rolling(w).mean().shift(1)
    feat[f'Roll_Std_{w}']  = feat['Close'].rolling(w).std().shift(1)

# RSI(14)
def compute_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    rs    = gain / loss.replace(0, np.nan)
    return 100 - 100 / (1 + rs)

feat['RSI_14'] = compute_rsi(feat['Close']).shift(1)

# MACD
ema12 = feat['Close'].ewm(span=12, adjust=False).mean()
ema26 = feat['Close'].ewm(span=26, adjust=False).mean()
feat['MACD'] = (ema12 - ema26).shift(1)

# Bollinger Band position (0 = lower band, 1 = upper band)
bb_mean = feat['Close'].rolling(20).mean()
bb_std  = feat['Close'].rolling(20).std()
feat['BB_position'] = ((feat['Close'] - (bb_mean - 2*bb_std)) / (4*bb_std + 1e-9)).shift(1)

# ATR(14)
tr = pd.concat([
    feat['High'] - feat['Low'],
    (feat['High'] - feat['Close'].shift(1)).abs(),
    (feat['Low']  - feat['Close'].shift(1)).abs(),
], axis=1).max(axis=1)
feat['ATR_14'] = tr.rolling(14).mean().shift(1)

# On-Balance Volume
obv_raw = (np.sign(feat['Close'].diff()) * feat['Volume']).fillna(0).cumsum()
feat['OBV'] = obv_raw.shift(1)

# Volume ratio vs 20-day average
feat['Volume_Ratio'] = (feat['Volume'] / feat['Volume'].rolling(20).mean()).shift(1)

# Calendar
feat['Year']       = feat.index.year
feat['Month']      = feat.index.month
feat['Day']        = feat.index.day
feat['Dayofweek']  = feat.index.dayofweek
feat['Is_weekend'] = (feat.index.dayofweek >= 5).astype(int)
feat['Quarter']    = feat.index.quarter

feat = feat.dropna()

FEATURES = [
    'Year','Month','Day','Dayofweek','Is_weekend','Quarter',
    'Lag1','Lag3','Lag7','Lag14','Lag30',
    'Roll_Mean_7','Roll_Mean_14','Roll_Mean_30',
    'Roll_Std_7','Roll_Std_14','Roll_Std_30',
    'RSI_14','MACD','BB_position','ATR_14','OBV','Volume_Ratio',
]

print(f"Feature matrix: {feat.shape[0]:,} rows × {len(FEATURES)} features")

## 3. Chronological train / test split

We hold out the **last ~18 months** (Jan 2025 → present) — includes the 2024–2025 bull run
peak and consolidation: a genuinely challenging out-of-sample test.


In [ ]:
split_date = '2025-01-01'

train_data = feat.loc[:split_date].iloc[:-1]
test_data  = feat.loc[split_date:]

X_train, y_train = train_data[FEATURES], train_data['Close']
X_test,  y_test  = test_data[FEATURES],  test_data['Close']

print(f"Train: {train_data.index.min().date()} → {train_data.index.max().date()}  ({len(train_data):,} rows)")
print(f"Test : {test_data.index.min().date()} → {test_data.index.max().date()}  ({len(test_data):,} rows)")

## 4. Shared evaluation function

In [ ]:
def evaluate(actual, predicted, name):
    actual    = np.asarray(actual,    dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    r2   = r2_score(actual, predicted)
    if len(actual) > 1:
        actual_dir = np.sign(np.diff(actual))
        pred_dir   = np.sign(predicted[1:] - actual[:-1])
        dir_acc = np.mean(actual_dir == pred_dir) * 100
    else:
        dir_acc = np.nan
    return {'Model': name, 'MAE': mae, 'RMSE': rmse,
            'MAPE_%': mape, 'R2': r2, 'DirAcc_%': dir_acc}

results     = []
predictions = {}   # store for ensemble + plotting

## 5. Naive baseline (`tomorrow = today`)

In [ ]:
naive_pred = test_data['Lag1'].values
results.append(evaluate(y_test, naive_pred, 'Naive (t-1)'))
predictions['Naive'] = naive_pred
print(results[-1])

## 6. XGBoost (tuned with TimeSeriesSplit)

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

param_dist_xgb = {
    'learning_rate':    [0.01, 0.05, 0.1],
    'max_depth':        [4, 6, 8],
    'n_estimators':     [100, 200, 300],
    'subsample':        [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'reg_alpha':        [0, 0.1, 1],
    'reg_lambda':       [1, 5, 10],
}

search_xgb = RandomizedSearchCV(
    xgb.XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1),
    param_distributions=param_dist_xgb,
    n_iter=20, cv=tscv,
    scoring='neg_mean_absolute_error', random_state=42, n_jobs=1,
)
search_xgb.fit(X_train, y_train)
best_xgb = search_xgb.best_estimator_
print('Best XGB params:', search_xgb.best_params_)

xgb_pred = best_xgb.predict(X_test)
predictions['XGBoost'] = xgb_pred
results.append(evaluate(y_test, xgb_pred, 'XGBoost'))
print(results[-1])

## 7. LightGBM *(new model)*

LightGBM uses **leaf-wise** tree growth (vs XGBoost's level-wise), making it faster and
often more accurate on tabular data. Key advantages:
- Native missing-value handling
- Built-in L1/L2 regularization + `min_child_samples` to prevent overfitting
- Histogram-based splits → far faster training on large datasets


In [ ]:
param_dist_lgb = {
    'learning_rate':     [0.01, 0.05, 0.1],
    'num_leaves':        [31, 63, 127],
    'n_estimators':      [100, 200, 300],
    'max_depth':         [-1, 6, 10],
    'subsample':         [0.7, 0.8, 1.0],
    'colsample_bytree':  [0.7, 0.8, 1.0],
    'reg_alpha':         [0, 0.1, 1.0],
    'reg_lambda':        [0, 1.0, 5.0],
    'min_child_samples': [20, 50, 100],
}

search_lgb = RandomizedSearchCV(
    lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
    param_distributions=param_dist_lgb,
    n_iter=20, cv=tscv,
    scoring='neg_mean_absolute_error', random_state=42, n_jobs=1,
)
search_lgb.fit(X_train, y_train)
best_lgb = search_lgb.best_estimator_
print('Best LGB params:', search_lgb.best_params_)

lgb_pred = best_lgb.predict(X_test)
predictions['LightGBM'] = lgb_pred
results.append(evaluate(y_test, lgb_pred, 'LightGBM'))
print(results[-1])

## 8. Stacking Ensemble: XGBoost + LightGBM → Ridge *(new)*

Two-level stack:
- **Level 0**: both models generate out-of-fold (OOF) predictions on training data via `cross_val_predict`.
- **Level 1**: a Ridge meta-learner learns optimal weights from OOF preds, then is applied to test predictions.


In [ ]:
# Out-of-fold predictions for meta-learner training
oof_xgb = cross_val_predict(best_xgb, X_train, y_train, cv=tscv)
oof_lgb = cross_val_predict(best_lgb, X_train, y_train, cv=tscv)

meta_X_train = np.column_stack([oof_xgb, oof_lgb])
meta_X_test  = np.column_stack([xgb_pred, lgb_pred])

meta_model = Ridge(alpha=1.0)
meta_model.fit(meta_X_train, y_train)
stack_pred = meta_model.predict(meta_X_test)

predictions['Stacking'] = stack_pred
results.append(evaluate(y_test, stack_pred, 'Stack (XGB+LGB)'))
print(results[-1])
print(f"\nMeta-learner weights  →  XGBoost: {meta_model.coef_[0]:.3f}  |  LightGBM: {meta_model.coef_[1]:.3f}")

## 9. Prophet (multi-step, for reference)

In [ ]:
p_train = train_data.reset_index()[['Date', 'Close']]
p_train.columns = ['ds', 'y']

m = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=True,
    changepoint_prior_scale=0.1,
)
m.fit(p_train)

future      = pd.DataFrame({'ds': test_data.index})
fc          = m.predict(future)
prophet_pred = fc['yhat'].values
predictions['Prophet'] = prophet_pred
results.append(evaluate(y_test, prophet_pred, 'Prophet'))
print(results[-1])

## 10. LSTM (1-step walk-forward, scaler fit on train only)

In [ ]:
SEQ   = 60
close = df_raw['Close']

scaler = MinMaxScaler()
train_close = close.loc[:split_date].iloc[:-1].values.reshape(-1, 1)
scaler.fit(train_close)
scaled_all  = scaler.transform(close.values.reshape(-1, 1)).flatten()

train_len = len(train_close)
Xtr, ytr = [], []
for i in range(SEQ, train_len):
    Xtr.append(scaled_all[i-SEQ:i]); ytr.append(scaled_all[i])
Xtr = np.array(Xtr).reshape(-1, SEQ, 1); ytr = np.array(ytr)

model_lstm = Sequential([
    Input((SEQ, 1)),
    LSTM(128, return_sequences=True), Dropout(0.2),
    LSTM(64),  Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1),
])
model_lstm.compile(optimizer='adam', loss='mse')
es = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1)
model_lstm.fit(Xtr, ytr, epochs=100, batch_size=32, validation_split=0.1, callbacks=[es], verbose=1)

In [ ]:
test_positions = [close.index.get_loc(d) for d in test_data.index]
seqs       = np.array([scaled_all[p-SEQ:p] for p in test_positions]).reshape(-1, SEQ, 1)
lstm_scaled = model_lstm.predict(seqs).flatten()
lstm_pred   = scaler.inverse_transform(lstm_scaled.reshape(-1, 1)).flatten()

predictions['LSTM'] = lstm_pred
results.append(evaluate(y_test, lstm_pred, 'LSTM'))
print(results[-1])

## 11. Results — fair comparison on the identical test window

In [ ]:
summary = pd.DataFrame(results).set_index('Model').round(2)
summary = summary.sort_values('MAE')

naive_mae = summary.loc['Naive (t-1)', 'MAE']
summary['vs_Naive_%'] = ((summary['MAE'] - naive_mae) / naive_mae * 100).round(1)

print(summary.to_string())
summary.style.background_gradient(subset=['MAE','RMSE','MAPE_%'], cmap='RdYlGn_r') \
             .background_gradient(subset=['DirAcc_%','R2'], cmap='RdYlGn')

## 12. Visualizations

In [ ]:
# Plot 1: Predictions vs Actual
colors = {'Naive':'grey','XGBoost':'royalblue','LightGBM':'darkorange',
          'Stacking':'crimson','Prophet':'mediumseagreen','LSTM':'purple'}

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(test_data.index, y_test.values, label='Actual', color='black', linewidth=2.5, zorder=5)
for name, preds in predictions.items():
    ax.plot(test_data.index, preds, label=name, alpha=0.75,
            linewidth=1.4, color=colors.get(name, 'tab:blue'))
ax.set_title('BTC-USD 1-day-ahead forecasts — test window (2025–2026)', fontsize=14)
ax.set_xlabel('Date'); ax.set_ylabel('Price (USD)')
ax.legend(loc='upper left', fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout(); plt.show()

In [ ]:
# Plot 2: Metric bars
fig, axs = plt.subplots(1, 4, figsize=(18, 5))
metrics = ['MAE', 'RMSE', 'MAPE_%', 'DirAcc_%']
better  = ['lower','lower','lower','higher']
pal = list(colors.values())

for ax, metric, b in zip(axs, metrics, better):
    vals = summary[metric].sort_values(ascending=(b == 'lower'))
    bars = ax.bar(vals.index, vals.values, color=pal[:len(vals)], edgecolor='white')
    ax.set_title(f'{metric}  ({b}=better)', fontsize=11)
    ax.tick_params(axis='x', rotation=38, labelsize=8)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                f'{val:.1f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout(); plt.show()

In [ ]:
# Plot 3: Rolling 30-day MAE
fig, ax = plt.subplots(figsize=(16, 5))
for name, preds in predictions.items():
    rolling_err = pd.Series(np.abs(y_test.values - preds), index=test_data.index).rolling(30).mean()
    ax.plot(test_data.index, rolling_err, label=name, alpha=0.85,
            linewidth=1.5, color=colors.get(name, 'tab:blue'))
ax.set_title('Rolling 30-day MAE — test window', fontsize=13)
ax.set_ylabel('MAE (USD)'); ax.set_xlabel('Date')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=10); plt.tight_layout(); plt.show()

In [ ]:
# Plot 4: Feature importance side-by-side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

pd.Series(best_xgb.feature_importances_, index=FEATURES).nlargest(15).sort_values() \
    .plot.barh(ax=ax1, color='royalblue', title='XGBoost — top 15 features')
pd.Series(best_lgb.feature_importances_, index=FEATURES).nlargest(15).sort_values() \
    .plot.barh(ax=ax2, color='darkorange', title='LightGBM — top 15 features')

for ax in (ax1, ax2):
    ax.set_xlabel('Importance')

plt.tight_layout(); plt.show()

## 13. How to read these results

| Metric | Interpretation |
|---|---|
| **MAE / RMSE** | Average dollar error — compare every model to **Naive (t-1)** |
| **MAPE %** | Scale-independent percentage error |
| **R²** | Variance explained — can be high even for a lag-follower |
| **DirAcc %** | % of days model correctly called up/down — 50% = coin flip |
| **vs Naive %** | MAE improvement (−) or degradation (+) over the baseline |

### Why LightGBM?
LightGBM's leaf-wise growth and histogram binning make it:
- **~3–10× faster** than XGBoost on training
- Often better MAE on tabular data with many features
- More regularization knobs (`min_child_samples`, `num_leaves`) to control overfitting

In the stacking ensemble, the Ridge meta-learner automatically up-weights whichever
base model is stronger, often beating both individually.

### Honest conclusion
Beating a naïve random-walk on a fair, leakage-free BTC backtest remains genuinely hard.
LightGBM + stacking represent the strongest attempt in this suite, with the new technical
features (RSI, MACD, ATR, OBV) providing incremental signal beyond raw price lags.
